# Open-weights rung - reliability probe on Colab (vLLM + Qwen3.6-27B)

Adds an **open-weights, white-box** model to the R90 ladder. Downloads **Qwen/Qwen3.6-27B** (dense, Apache-2.0, native **262,144**-token window - verified on the HF card 2026-06-22), serves it through vLLM's **OpenAI-compatible** endpoint (the probe's OpenAI adapter then works with only an env-var swap), and runs a **near-only smoke test** as the hard go/no-go before any GPU-expensive sweep.

**Footguns this notebook handles for you (all card-verified):**
- **VRAM:** 27B in BF16 is ~54 GB of weights - needs an **80 GB A100**. It will NOT fit a 40 GB A100. Cell 2 checks.
- **No YaRN:** the 262k native window already exceeds every fill we use, so `--rope-scaling` is left **OFF** (the card warns static YaRN degrades shorter texts - which would corrupt the near gate).
- **Thinking OFF:** Qwen3.6 thinks by default and has no `/no_think` switch; we force `enable_thinking=False` per request so it matches the non-reasoning API panel (gpt-3.5 / gpt-4o-mini / Haiku / Sonnet) and the tool call is not starved by a think block.
- **Tool parser:** `--tool-call-parser qwen3_coder` (the probe uses native function-calling).


In [ ]:
# --- config: HIGH-FILL continuation (8k-64k already banked in colab_runs/ from 22 Jun) ---
MODEL = 'Qwen/Qwen3.6-27B'   # dense, Apache-2.0, native 262,144 window
MODEL_DIR = '/content/model'
SERVED = 'open-model'        # keep the label to EXTEND yesterday's curve. If you fall back to fp8
                             # KV below, change to 'open-model-fp8kv' so the dtype switch is tracked.
MAX_LEN = 262144             # FULL native window now (no YaRN) -> reach for the knee (flat to ctx ~95k).
KV_DTYPE = 'auto'            # 'auto' = BF16 KV (clean). If serve fails with a "KV cache too small"
                             # error, set 'fp8' (halves KV, fits 262k) + relabel SERVED; report as a confound.
GMEM = 0.95                  # squeeze KV headroom for the big window
HI_FILLS = '96000 144000 184000'   # HIGH fills only (ctx ~130k/195k/248k); low fills done yesterday.
REPO_URL = 'https://github.com/R1ch1k/agent-reliability.git'

In [ ]:
# --- 0. confirm the hardware (decides feasibility) ---
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
print()
print('NEED ~80 GB: Qwen3.6-27B in BF16 is ~54 GB of weights; it does NOT fit a 40 GB A100.')
print('If memory.total is ~40 GB: Runtime > Change runtime type > A100 (80 GB), or fall back to a')
print('smaller CURRENT dense model at FULL precision. Do NOT 4-bit a 27B - quantization is the')
print('confound this white-box rung exists to avoid (near>=0.97 also guards against a broken quant).')


In [ ]:
# --- 1. install vLLM matched to THIS runtime's CUDA ---
# vLLM's PyPI wheel is built for ONE CUDA (currently 13); Colab ships torch for a different
# CUDA (e.g. cu128), so a plain `pip install vllm` leaves vllm._C unable to find libcudart
# -> ImportError: libcudart.so.XX. Install via uv with --torch-backend=auto + --reinstall so
# torch + vLLM + CUDA libs become ONE consistent stack matched to the GPU driver (~2-3 min).
!pip install -q uv huggingface_hub requests
!uv pip install --system --reinstall 'vllm>=0.19' --torch-backend=auto
!python -c "import vllm; print('vLLM', vllm.__version__); import vllm._C; print('vllm._C loads OK')"

In [ ]:
# --- 2. download weights (point local_dir at a mounted Drive path to persist across sessions) ---
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=MODEL_DIR)   # Apache-2.0, ungated (no token)
print('downloaded to', MODEL_DIR)


In [ ]:
# --- 3. serve at the FULL native window (no YaRN). Watch the log for the KV-cache capacity. ---
import subprocess, time, requests
args = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_DIR,
    '--served-model-name', SERVED,
    '--max-model-len', str(MAX_LEN),
    '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder',  # Qwen3.6 tool parser (card)
    '--enable-prefix-caching',          # reuse the shared manual prefix within a run
    '--gpu-memory-utilization', str(GMEM),
    '--port', '8000',
]
if KV_DTYPE != 'auto':
    args += ['--kv-cache-dtype', KV_DTYPE]   # fp8 halves KV memory to reach the full window (CONFOUND)
logf = open('/content/vllm.log', 'w')
srv = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for _ in range(360):                    # up to 30 min: 27B download-to-GPU load is slow
    try:
        if requests.get('http://localhost:8000/health').ok:
            ok = True; print('server UP  (KV dtype =', KV_DTYPE, '| max_model_len =', MAX_LEN, ')'); break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print('server did NOT come up - last log lines:')
    print(''.join(open('/content/vllm.log').readlines()[-40:]))
    print("If the log says the KV cache cannot hold max_model_len: either LOWER MAX_LEN, or set")
    print("KV_DTYPE='fp8' and SERVED='open-model-fp8kv' in the config cell, then re-run this cell.")

In [ ]:
# --- 4. point the OpenAI SDK at the local server + force NON-thinking mode ---
import os, json
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'EMPTY'   # vLLM needs no auth
# Qwen3.6 THINKS BY DEFAULT and has no /no_think switch. The API panel was all non-reasoning,
# so for a capability-controlled comparison we force thinking OFF. The probe reads this env var
# (PROBE_OPENAI_EXTRA_BODY) and passes it as extra_body on every call.
os.environ['PROBE_OPENAI_EXTRA_BODY'] = json.dumps({'chat_template_kwargs': {'enable_thinking': False}})

from openai import OpenAI
r = OpenAI().chat.completions.create(
    model=SERVED,
    messages=[{'role': 'user', 'content': 'reply with the single word OK'}],
    max_tokens=16,
    extra_body=json.loads(os.environ['PROBE_OPENAI_EXTRA_BODY']),
)
print(repr(r.choices[0].message.content))   # expect 'OK' with NO think block


In [ ]:
# --- 5. clone harness + NEAR-only smoke at the HIGH fills (validates the new window/KV first) ---
!git clone -q $REPO_URL /content/probe
%cd /content/probe
# OPENAI_BASE_URL / OPENAI_API_KEY / PROBE_OPENAI_EXTRA_BODY from cell 4 propagate into this subprocess.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near --fills $HI_FILLS --runs 3 --needles 3 --depth 0.5 --calib 0

## Go / no-go (high-fill continuation)

- **near holds ~1.0 at the high fills** -> the big window + KV dtype are sound; run the full sweep.
- **near drops** -> the large window or fp8 KV is degrading retrieval. If you switched to fp8 KV, that
  is the confound biting: either lower MAX_LEN and stay BF16, or report the fp8 caveat explicitly.

Cell 6 runs near+distance at the high fills. Combined with yesterday's banked 8k-64k curve
(`colab_runs/dist_results_20260622_210615.jsonl`) this spans ctx ~12k -> ~248k.

**Record the confound (not logged by the harness):** Qwen3.6-27B, BF16 weights, **KV dtype = whatever
KV_DTYPE you used**, max-model-len, YaRN OFF. The high-ctx runs are SLOW (huge prefill) — drop `--runs`
to 5 if you want it faster.

In [ ]:
# --- 6. FULL near+distance sweep at the HIGH fills (runs=8; high-ctx runs are slow, ~2-3h). ---
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills $HI_FILLS --runs 8 --needles 3 --depth 0.5

In [ ]:
# --- 7. DISENTANGLING pass — DEFERRED: only run AFTER cell 6 if distance actually bent. ---
# --padding inert + --fixed-needle-seed isolate raw context-LENGTH from search-space size. On a FLAT
# curve there is nothing to disentangle, so SKIP this if distance stayed ~1.0. runs=8 to save GPU.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills $HI_FILLS --runs 8 --needles 3 --depth 0.5 --padding inert --fixed-needle-seed 7

## Save + combine onto the ladder

Download (cell 8 below), then on your machine:
1. Put today's high-fill `dist_results_*.jsonl` next to yesterday's `colab_runs/dist_results_20260622_210615.jsonl` (8k-64k).
2. **If both are BF16-KV** (KV_DTYPE stayed 'auto'): one consistent `open-model` curve — add both files to `canonical_manifest.txt`, add `'open-model'` to the `order` list in `analyze_curves.py`, then re-run `analyze_curves.py` + `make_figures.py`.
3. **If today used fp8 KV** ('open-model-fp8kv'): keep it as a separate, confound-flagged high-fill arm. The near control + the BF16/fp8 behaviour at the overlap is your fp8-vs-BF16 cross-check.
4. The **inert-padded** disentangling file (if you ran cell 7) stays OUT of `canonical_manifest.txt`.

In [ ]:
# --- 8. zip + download all results (Colab is ephemeral) ---
from google.colab import files
import glob, zipfile
out = '/content/probe_results.zip'
with zipfile.ZipFile(out, 'w') as z:
    for f in glob.glob('/content/probe/dist_results_*.jsonl'):
        z.write(f, arcname=f.split('/')[-1])
    z.write('/content/vllm.log', arcname='vllm.log')
files.download(out)
print('zipped + downloading', out)
